Partie A — Questions à traiter pour chaque algorithme (1 à 29)
**texte en gras**

In [8]:
# Q0 — fonction utilitaire réutilisable pour les 29 algorithmes
from sklearn.datasets import make_regression, make_classification, make_blobs, make_moons

def afficher_donnees(X, y, titre="Données synthétiques", tache="classification"):
    """Affiche le DataFrame et une visualisation 2D des données synthétiques."""
    df = pd.DataFrame(X, columns=[f"x{j}" for j in range(X.shape[1])])
    df["cible"] = y
    print(f"--- {titre} : {X.shape[0]} observations, {X.shape[1]} variables ---")
    display(df.head())
    display(df.describe().round(3))
    if X.shape[1] >= 2:
        plt.figure(figsize=(5,4))
        if tache == "regression":
            plt.scatter(X[:,0], y, s=15, alpha=.7)
            plt.xlabel("x0"); plt.ylabel("cible")
        else:
            plt.scatter(X[:,0], X[:,1], c=y, cmap="tab10", s=15)
            plt.xlabel("x0"); plt.ylabel("x1")
        plt.title(titre); plt.tight_layout(); plt.show()

PARTIE **B**

In [2]:
# B.0 — imports communs Partie B
!pip -q install scikit-learn xgboost umap-learn gymnasium openpyxl

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import sklearn, warnings
warnings.filterwarnings("ignore")
GRAINE = 42
np.random.seed(GRAINE)
print("scikit-learn", sklearn.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)


scikit-learn 1.6.1 | numpy 2.4.6 | pandas 2.2.3


In [4]:
# Q5 — tableau de traçabilité (à compléter avec votre date de consultation réelle)
from datetime import date
tracabilite = pd.DataFrame([
    {"Dataset":"Default of Credit Card Clients","Source":"UCI ML Repository",
     "URL":"https://archive.ics.uci.edu/dataset/350",
     "Licence":"CC BY 4.0","Taille":"30 000 × 24",
     "Cible":"défaut de paiement (0/1)","Date consultation":date.today().isoformat()},
    {"Dataset":"Online Retail","Source":"UCI ML Repository",
     "URL":"https://archive.ics.uci.edu/dataset/352",
     "Licence":"CC BY 4.0","Taille":"~541 909 lignes",
     "Cible":"aucune (segmentation / panier)","Date consultation":date.today().isoformat()},
])
display(tracabilite)
# Q5 — téléchargement du dataset UCI "Default of Credit Card Clients" (classification)
import urllib.request, zipfile, io, os
URL_CREDIT = "https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip"
if not os.path.exists("credit.zip"):
    urllib.request.urlretrieve(URL_CREDIT, "credit.zip")
with zipfile.ZipFile("credit.zip") as z:
    nom_xls = [f for f in z.namelist() if f.lower().endswith(".xls")][0]
    df_credit = pd.read_excel(io.BytesIO(z.read(nom_xls)), header=1)
print("Forme:", df_credit.shape)
display(df_credit.head())
print("Colonnes:", list(df_credit.columns))
# Q5 — téléchargement du dataset UCI "Online Retail" (clustering / segmentation)
URL_RETAIL = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
if not os.path.exists("retail.zip"):
    urllib.request.urlretrieve(URL_RETAIL, "retail.zip")
with zipfile.ZipFile("retail.zip") as z:
    nom_xls = [f for f in z.namelist() if f.lower().endswith(".xlsx")][0]
    df_retail = pd.read_excel(io.BytesIO(z.read(nom_xls)))
print("Forme:", df_retail.shape)
display(df_retail.head())


,Dataset,Source,URL,Licence,Taille,Cible,Date consultation
0,Default of Credit Card Clients,UCI ML Repository,https://archive.ics.uci.edu/dataset/350,CC BY 4.0,30 000 × 24,défaut de paiement (0/1),2026-11-10
1,Online Retail,UCI ML Repository,https://archive.ics.uci.edu/dataset/352,CC BY 4.0,~541 909 lignes,aucune (segmentation / panier),2026-11-10


Forme: (30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


Colonnes: ['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default payment next month']
Forme: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [5]:
# Q6 — audit des données : manquants, types, doublons, déséquilibre
def audit(df, cible=None):
    print("=== Valeurs manquantes ===")
    print(df.isna().sum()[df.isna().sum() > 0] if df.isna().any().any() else "aucune")
    print("=== Types ===")
    print(df.dtypes.value_counts())
    print("=== Doublons ===", df.duplicated().sum())
    if cible:
        print("=== Déséquilibre de classes ===")
        print(df[cible].value_counts(normalize=True).round(3))

audit(df_credit, cible="default payment next month")
audit(df_retail)
# Q6 — valeurs extrêmes (règle IQR) sur variables numériques clés
def extremes_iqr(df, cols):
    rapport = {}
    for c in cols:
        q1, q3 = df[c].quantile([.25,.75]); iqr = q3-q1
        rapport[c] = int(((df[c] < q1-1.5*iqr) | (df[c] > q3+1.5*iqr)).sum())
    return pd.Series(rapport, name="nb_extrêmes")

display(extremes_iqr(df_credit, ["LIMIT_BAL","BILL_AMT1","PAY_AMT1"]))
# Q6 — pipeline anti-fuite : imputation + encodage + standardisation ajustés sur train
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def preparer(df, cible, cols_num, cols_cat):
    X = df.drop(columns=[cible]); y = df[cible]
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc",  StandardScaler())]), cols_num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh",  OneHotEncoder(handle_unknown="ignore"))]), cols_cat),
    ])
    return train_test_split(X, y, test_size=.25, stratify=y, random_state=GRAINE), pre

# exemple sur crédit : cible binaire à créer
df_credit["defaut"] = (df_credit["default payment next month"] == 1).astype(int)
cols_num = ["LIMIT_BAL","AGE","BILL_AMT1","BILL_AMT2","PAY_AMT1","PAY_AMT2"]
cols_cat = ["SEX","EDUCATION","MARRIAGE"]
(Xc_tr, Xc_te, yc_tr, yc_te), pre_credit = preparer(df_credit, "defaut", cols_num, cols_cat)
print("Train:", Xc_tr.shape, "| Test:", Xc_te.shape)
print("Taux défaut train:", yc_tr.mean().round(3), "| test:", yc_te.mean().round(3))


=== Valeurs manquantes ===
aucune
=== Types ===
int64    25
Name: count, dtype: int64
=== Doublons === 0
=== Déséquilibre de classes ===
default payment next month
0    0.779
1    0.221
Name: proportion, dtype: float64
=== Valeurs manquantes ===
Description      1454
CustomerID     135080
dtype: int64
=== Types ===
object            4
float64           2
int64             1
datetime64[ns]    1
Name: count, dtype: int64
=== Doublons === 5268


,nb_extrêmes
LIMIT_BAL,167
BILL_AMT1,2400
PAY_AMT1,2745


Train: (22500, 25) | Test: (7500, 25)
Taux défaut train: 0.221 | test: 0.221


In [10]:
# Q7 — clustering : critères internes + évaluation externe (étiquettes réelles)
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

wine = load_wine(); Xw = StandardScaler().fit_transform(wine.data); yw = wine.target
Xw2 = PCA(2, random_state=GRAINE).fit_transform(Xw)  # pour visu seulement

algos_clu = {
    "K-means (réf.)":  KMeans(3, n_init=15, random_state=GRAINE),
    "Ward (réf.)":     AgglomerativeClustering(n_clusters=3, linkage="ward"),
    "GMM (avancé)":    GaussianMixture(n_components=3, random_state=GRAINE),
    "DBSCAN (avancé)": DBSCAN(eps=.7, min_samples=6),
}
res_clu = {}
for nom, m in algos_clu.items():
    lab = m.fit_predict(Xw) if hasattr(m, "fit_predict") else m.fit(Xw).predict(Xw)
    masque = lab != -1
    if len(set(lab[masque])) > 1:
        res_clu[nom] = {"Silhouette (interne)": silhouette_score(Xw[masque], lab[masque]),
                        "ARI (externe)": adjusted_rand_score(yw[masque], lab[masque]),
                        "NMI (externe)": normalized_mutual_info_score(yw[masque], lab[masque]),
                        "Groupes": len(set(lab)-{-1}),
                        "Bruit": int((lab==-1).sum())}
display(pd.DataFrame(res_clu).T.round(3))


,Silhouette (interne),ARI (externe),NMI (externe),Groupes,Bruit
K-means (réf.),0.285,0.897,0.876,3.0,0.0
Ward (réf.),0.277,0.790,0.786,3.0,0.0
GMM (avancé),0.285,0.897,0.876,3.0,0.0
